# Football Match Prediction & Betting Market Analysis

**Author:** Charlie Findley  
**Project type:** Data analysis / machine learning / sports analytics  
**Dataset:** English Premier League 2024–25 match results and bookmaker odds from Football-Data.co.uk

## Project Goal

The aim of this project is to build machine learning models that predict Premier League match outcomes and compare those predictions against bookmaker odds.

The central question is:

> **Can engineered football performance features improve predictive accuracy beyond the information already contained in Bet365 betting odds?**

This is both a prediction problem and a market-efficiency question. If football-specific features improve on bookmaker-derived features, that would suggest that the model is capturing extra signal not fully priced into the market. If they do not, that suggests the bookmaker odds already summarise most of the useful available information.

## Main Findings

The strongest benchmark in this project was the simple **Bet365 favourite rule**, which predicts the outcome with the lowest Bet365 odds. The machine learning models trained on bookmaker-derived features, football-specific features, and combined features did not clearly outperform this baseline.

The final conclusion is that, on this dataset, **Bet365 odds are more predictive than the engineered football features**, and adding football-specific features did not provide evidence of additional predictive value beyond the betting market.

In [10]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", 120)

## 1. Dataset

The dataset contains match statistics, match results, and bookmaker odds for the English Premier League 2024–25 season.

The notebook expects the data file to be available locally as either:

- `historical_football_data.csv`, or
- `football.csv`

The modelling uses only variables that would have been available before kick-off. Match statistics such as full-time goals, shots, cards, corners and half-time results are used for exploration and feature construction, but are not directly used as model inputs for pre-match prediction.

In [11]:
possible_files = [
    "historical_football_data.csv",
    "football.csv",
    "data/historical_football_data.csv",
    "data/football.csv"
]

data_path = None

for file in possible_files:
    if os.path.exists(file):
        data_path = file
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find the football dataset. Please place the CSV file in the same folder as this notebook "
        "and name it 'historical_football_data.csv' or 'football.csv'."
    )

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
df.head()

FileNotFoundError: Could not find the football dataset. Please place the CSV file in the same folder as this notebook and name it 'historical_football_data.csv' or 'football.csv'.

In [ ]:
print("Dataset shape:", df.shape)

display(df.info())
display(df.describe(include="all"))

# 2. Understanding the Variables

Before any modelling can be performed, it is important to understand the meaning of each variable contained within the dataset. The dataset contains information relating to match outcomes, team performance statistics, and bookmaker odds for Premier League fixtures during the 2024–25 season.

The variables can be grouped into several categories:

---

## 2.1 Match Information

| Variable | Description |
|-----------|------------|
| Div | League division identifier |
| Date | Date on which the match was played |
| Time | Kick-off time |
| HomeTeam | Home team |
| AwayTeam | Away team |

These variables identify the fixture itself and provide contextual information about when and where the match occurred.

---

## 2.2 Match Outcome Variables

| Variable | Description |
|-----------|------------|
| FTHG | Full-Time Home Goals |
| FTAG | Full-Time Away Goals |
| FTR | Full-Time Result (H = Home Win, D = Draw, A = Away Win) |
| HTHG | Half-Time Home Goals |
| HTAG | Half-Time Away Goals |
| HTR | Half-Time Result |

These variables describe the final outcome of the match and are often used as target variables for predictive modelling.

### Target Variable

For this project, the primary target variable is:

**FTR (Full-Time Result)**

where:

- H = Home Win
- D = Draw
- A = Away Win

The objective of the machine learning model will be to predict this result before a match takes place.

---

## 2.3 Shooting Statistics

| Variable | Description |
|-----------|------------|
| HS | Home Team Shots |
| AS | Away Team Shots |
| HST | Home Team Shots on Target |
| AST | Away Team Shots on Target |

These variables measure attacking performance and provide an indication of how threatening each team was during the match.

Generally, teams producing more shots and more shots on target are expected to have a higher probability of winning.

---

## 2.4 Discipline Statistics

| Variable | Description |
|-----------|------------|
| HF | Home Team Fouls Committed |
| AF | Away Team Fouls Committed |
| HC | Home Team Corners |
| AC | Away Team Corners |
| HY | Home Team Yellow Cards |
| AY | Away Team Yellow Cards |
| HR | Home Team Red Cards |
| AR | Away Team Red Cards |

These statistics provide information regarding discipline, aggression, and territorial dominance.

For example:

- High corner counts may indicate sustained attacking pressure.
- Red cards can significantly impact match outcomes.
- Yellow cards may reflect aggressive defensive play.

---

## 2.5 Bookmaker Odds

The dataset contains odds from multiple bookmakers.

Examples include:

| Prefix | Bookmaker |
|----------|-----------|
| B365 | Bet365 |
| BW | Bet&Win |
| IW | Interwetten |
| PS | Pinnacle Sports |
| WH | William Hill |
| VC | VC Bet |
| Max | Maximum Odds Available |
| Avg | Average Market Odds |

For each bookmaker, three odds are typically provided:

| Variable Example | Description |
|-----------------|------------|
| B365H | Bet365 Home Win Odds |
| B365D | Bet365 Draw Odds |
| B365A | Bet365 Away Win Odds |

The same pattern applies to other bookmakers.

---

## 4.6 Over/Under Goal Market Odds

Many bookmakers also provide odds on whether the total number of goals scored exceeds a threshold.

Examples:

| Variable | Description |
|-----------|------------|
| B365>2.5 | Odds for Over 2.5 Goals |
| B365<2.5 | Odds for Under 2.5 Goals |

These odds contain valuable information about market expectations regarding match scoring.

---

## 4.7 Asian Handicap Markets

The dataset also contains Asian Handicap betting lines.

Examples:

| Variable | Description |
|-----------|------------|
| AHh | Asian Handicap Line |
| B365AHH | Home Handicap Odds |
| B365AHA | Away Handicap Odds |

Asian Handicap markets attempt to balance stronger and weaker teams by applying a virtual goal advantage or disadvantage.

These variables may contain useful information about perceived team strength.

---

## 4.8 Variables and Data Leakage

A key consideration when building predictive models is avoiding **data leakage**.

Data leakage occurs when information that would not have been known before kick-off is used to predict the match outcome.

Examples of variables that cannot be used directly for pre-match prediction include:

- FTHG
- FTAG
- HTHG
- HTAG
- HS
- AS
- HST
- AST
- HY
- AY
- HR
- AR

These statistics are generated during or after the match and would not be available when making a prediction beforehand.

Using these variables would produce unrealistically strong model performance.

---

## 4.9 Initial Modelling Considerations

At this stage, the dataset contains both:

### Information Available Before Kick-Off

- Team names
- Match date
- Bookmaker odds
- Betting market information

### Information Available After Kick-Off

- Goals scored
- Shots
- Cards
- Corners
- Match statistics

Only information available before the match should be used when constructing a realistic predictive model.

The next stage of the project is to identify which variables can legitimately be used as predictive features and which variables must be excluded to avoid data leakage.

# 3. Feature Selection and Data Leakage

The objective of this project is to predict the full-time result of a Premier League match before kick-off.

To ensure a realistic prediction setting, only information available before the match begins may be used as input features.

Any variables generated during or after the match are excluded to avoid data leakage.

## Target Variable

The target variable for this project is:

**FTR (Full Time Result)**

Possible values:

- H = Home Win
- D = Draw
- A = Away Win

The machine learning models developed in this project will attempt to predict FTR before the match takes place.

## Variables Excluded Due to Data Leakage

The following variables are generated during the match and would not be available before kick-off.

These variables are therefore excluded from model training.

In [ ]:
leakage_columns = [
    "FTHG","FTAG","HTHG","HTAG","HTR",
    "HS","AS","HST","AST",
    "HF","AF",
    "HC","AC",
    "HY","AY",
    "HR","AR"
]

## Valid Pre-Match Features

The remaining variables are available before kick-off and may therefore be used as predictors.

# 4. Exploratory Data Analysis (EDA)

Before building predictive models, it is important to explore the dataset and understand the structure of the data.

## Missing Values

The following table shows the number of missing observations in each column.

In [ ]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

### Missing Values

The majority of missing observations occur within bookmaker-specific odds columns, particularly Bet&Win and William Hill odds.

These variables are not essential to the predictive modelling process because equivalent information is available from several other bookmakers, including Bet365, Pinnacle, and market average odds.

Removing rows with missing values would unnecessarily reduce the dataset size. Therefore, these columns will be excluded from further analysis, allowing all 380 matches to be retained.

The remaining variables contain either no missing values or only a very small number of missing observations.

In [ ]:
print(df.shape)

df_clean = df.drop(columns=[
    "BWH","BWD","BWA",
    "BWCH","BWCD","BWCA",
    "WHH","WHD","WHA",
    "WHCH","WHCD","WHCA"
])

print(df_clean.shape)

## Distribution of Match Outcomes

The target variable for this project is FTR (Full Time Result), which records whether a match ended in:

- H = Home Win
- D = Draw
- A = Away Win

Understanding the distribution of outcomes is important because highly imbalanced classes can affect model performance and evaluation.

In [ ]:
result_counts = df["FTR"].value_counts()

print(result_counts)

result_counts.plot(kind="bar")

plt.title("Distribution of Match Outcomes")
plt.xlabel("Result")
plt.ylabel("Number of Matches")

plt.show()

The distribution of match outcomes shows that home wins occur more frequently than draws and away wins.

## Goal Distributions

Goals scored by home and away teams provide insight into the attacking characteristics of Premier League matches.

In [ ]:
print("Home Goals")
print(df["FTHG"].describe())

print("\nAway Goals")
print(df["FTAG"].describe())

In [ ]:
plt.figure(figsize=(10,4))

plt.hist(df["FTHG"], bins=10)

plt.title("Distribution of Home Goals")
plt.xlabel("Goals")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.hist(df["FTAG"], bins=10)

plt.title("Distribution of Away Goals")
plt.xlabel("Goals")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print("Average Home Goals:", df["FTHG"].mean())
print("Average Away Goals:", df["FTAG"].mean())

Home teams score more goals on average than away teams.

The distributions are right-skewed, with most matches featuring between zero and three goals per team, while very high-scoring performances are relatively rare.

This observation is consistent with the home advantage previously identified in the match outcome analysis.

## Bookmaker Odds Analysis

Bookmaker odds reflect the market's expectations regarding match outcomes.

Lower odds correspond to outcomes that bookmakers consider more likely.

This analysis investigates the distributions of home win, draw and away win odds from Bet365.

In [ ]:
bet365_cols = ["B365H", "B365D", "B365A"]
df[bet365_cols].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))

axes[0].hist(df["B365H"], bins=20)
axes[0].set_title("Home Win Odds")
axes[0].set_xlabel("Odds")

axes[1].hist(df["B365D"], bins=20)
axes[1].set_title("Draw Odds")
axes[1].set_xlabel("Odds")

axes[2].hist(df["B365A"], bins=20)
axes[2].set_title("Away Win Odds")
axes[2].set_xlabel("Odds")

plt.tight_layout()
plt.show()

In [ ]:
df.groupby("FTR")[["B365H", "B365D", "B365A"]].mean().round(2)

Comparing average odds across actual match outcomes provides an indication of whether bookmaker expectations align with realised results.

If bookmakers are effective, matches ending in home wins should generally have lower home-win odds than matches ending in draws or away wins.

In [ ]:
def favourite(row):
    odds = {
        "H": row["B365H"],
        "D": row["B365D"],
        "A": row["B365A"]
    }
    return min(odds, key=odds.get)

df["Favourite"] = df.apply(favourite, axis=1)

In [ ]:
df["Favourite"].value_counts()

In [ ]:
favourite_accuracy = (
    df["Favourite"] == df["FTR"]
).mean()

print(f"Favourite Accuracy: {favourite_accuracy:.3f}")

## Correlation Analysis

Correlation analysis helps identify relationships between key numerical variables in the dataset.


In [ ]:
corr_cols = [
    "FTHG", "FTAG",
    "HS", "AS",
    "HST", "AST",
    "HC", "AC",
    "HY", "AY",
    "B365H", "B365D", "B365A"
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10,8))

plt.imshow(corr_matrix)

plt.colorbar()

plt.xticks(
    range(len(corr_cols)),
    corr_cols,
    rotation=90
)

plt.yticks(
    range(len(corr_cols)),
    corr_cols
)

plt.title("Correlation Heatmap")

plt.tight_layout()
plt.show()

# 5. Bet365 Feature Engineering

In [ ]:
df["Prob_H"] = 1 / df["B365H"]
df["Prob_D"] = 1 / df["B365D"]
df["Prob_A"] = 1 / df["B365A"]

total = df["Prob_H"] + df["Prob_D"] + df["Prob_A"]

df["Prob_H"] /= total
df["Prob_D"] /= total
df["Prob_A"] /= total

In [ ]:
df["HomeFav"] = (df["Favourite"] == "H").astype(int)
df["AwayFav"] = (df["Favourite"] == "A").astype(int)

In [ ]:
def odds_gap(row):
    odds = sorted([row["B365H"], row["B365D"], row["B365A"]])
    return odds[1] - odds[0]

df["OddsGap"] = df.apply(odds_gap, axis=1)

In [ ]:
df["MaxProb"] = df[["Prob_H","Prob_D","Prob_A"]].max(axis=1)

## 6. Bet365 Baseline Prediction Models

The first set of models uses only information derived from Bet365 betting odds. This creates a bookmaker-based benchmark against which football-specific features can later be compared.

Three bookmaker-based approaches are evaluated:

1. **Favourite rule**: predict the outcome with the lowest Bet365 odds.
2. **Logistic Regression** using Bet365-derived features.
3. **Random Forest** using Bet365-derived features.

The train-test split is stratified so that the outcome distribution is similar in both the training and test sets.

In [ ]:
bet365_features = [
    "Prob_H",
    "Prob_D",
    "Prob_A",
    "HomeFav",
    "AwayFav",
    "OddsGap",
    "MaxProb"
]

X_bet365 = df[bet365_features]
y_bet365_labels = df["FTR"]

X_train, X_test, y_train_labels, y_test_labels = train_test_split(
    X_bet365,
    y_bet365_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_bet365_labels
)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train_labels)
y_test = encoder.transform(y_test_labels)

print("Training matches:", X_train.shape[0])
print("Testing matches:", X_test.shape[0])
print("Target classes:", list(encoder.classes_))

### Logistic Regression

Logistic Regression is used as a simple baseline model.

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, log_preds)
log_f1 = f1_score(y_test, log_preds, average="macro")

print("Bet365 Logistic Regression Accuracy:", round(log_accuracy, 4))
print("Bet365 Logistic Regression Macro F1:", round(log_f1, 4))

### Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds, average="macro")

print("Bet365 Random Forest Accuracy:", round(rf_accuracy, 4))
print("Bet365 Random Forest Macro F1:", round(rf_f1, 4))

In [ ]:
fav_test_preds_labels = df.loc[X_test.index, "Favourite"]
fav_test_preds = encoder.transform(fav_test_preds_labels)

fav_accuracy = accuracy_score(y_test, fav_test_preds)
fav_f1 = f1_score(y_test, fav_test_preds, average="macro")

print("Bet365 Favourite Rule Accuracy:", round(fav_accuracy, 4))
print("Bet365 Favourite Rule Macro F1:", round(fav_f1, 4))

In [ ]:
bet365_results = pd.DataFrame({
    "Feature Set": [
        "Bet365 Odds",
        "Bet365 Odds",
        "Bet365 Odds"
    ],
    "Model": [
        "Favourite Rule",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        fav_accuracy,
        log_accuracy,
        rf_accuracy
    ],
    "Macro F1": [
        fav_f1,
        log_f1,
        rf_f1
    ]
})

bet365_results.sort_values("Accuracy", ascending=False)

## Bet365 Baseline Model Results

The bookmaker favourite rule is a strong benchmark because it directly uses the market's most likely outcome. The machine learning models use more detailed transformations of the same odds, such as normalised implied probabilities and the gap between the favourite and second-favourite outcome.

If the machine learning models cannot beat the favourite rule, this suggests that the extra transformations of the odds do not add much predictive value beyond the basic market ranking of outcomes.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(bet365_results["Model"], bet365_results["Accuracy"])
plt.title("Bet365 Baseline Model Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

The bookmaker favourite rule achieved the highest predictive accuracy among the Bet365-based baseline methods.

This suggests that much of the predictive information contained within the betting odds is already captured by the favourite outcome itself. The machine learning models were unable to clearly improve upon this benchmark when only bookmaker-derived features were used.

This result is not unexpected, as bookmaker odds incorporate extensive information regarding team strength, injuries, historical performance, betting market activity and expert judgement.

The next stage of the project investigates whether football-specific features can provide additional predictive value beyond that already contained within the betting market.

## 7. Football Feature Engineering

The previous models used only bookmaker-derived predictors. I now construct football-specific features using information that would have been available before each match.

The aim is to test whether recent team performance, league position, goal difference and home/away strength add predictive value beyond the Bet365 odds.

To avoid data leakage, each feature is calculated before updating a team's record with the current match result. This means each match only has access to information from previous matches, not from itself or future fixtures.

In [ ]:

df = df.copy()


df["MatchDate"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


if "Time" in df.columns:
    df["KickoffTime"] = df["Time"].fillna("00:00").astype(str)
    df["DateTime"] = pd.to_datetime(
        df["MatchDate"].dt.strftime("%Y-%m-%d") + " " + df["KickoffTime"],
        errors="coerce"
    )
else:
    df["DateTime"] = df["MatchDate"]

df["DateTime"] = df["DateTime"].fillna(df["MatchDate"])


df = df.sort_values(["DateTime", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

In [ ]:
teams = sorted(set(df["HomeTeam"]).union(set(df["AwayTeam"])))

# Create a dictionary to store each team's running statistics
team_stats = {}

for team in teams:
    team_stats[team] = {
        "played": 0,
        "points": 0,
        "gf": 0,
        "ga": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "recent_points": [],
        "recent_gf": [],
        "recent_ga": [],
        "recent_gd": [],
        "recent_home_points": [],
        "recent_away_points": []
    }


def last_n_total(values, n=5):
    """
    Returns the total of the last n values.
    If the team has not played yet, return 0.
    """
    if len(values) == 0:
        return 0
    return sum(values[-n:])


def last_n_average(values, n=5):
    """
    Returns the average of the last n values.
    If the team has not played yet, return 0.
    """
    if len(values) == 0:
        return 0
    return sum(values[-n:]) / len(values[-n:])


def win_rate(stats):
    """
    Returns a team's win rate before the match.
    If the team has not played yet, return 0.
    """
    if stats["played"] == 0:
        return 0
    return stats["wins"] / stats["played"]


def get_league_positions(team_stats):
    """
    Calculates the league position before each match.

    Teams are ranked by:
    1. Points
    2. Goal difference
    3. Goals scored

    Tied teams receive the average of their tied positions.
    For example, if two teams are tied for 1st, both receive position 1.5.
    """
    table = []

    for team, stats in team_stats.items():
        table.append({
            "Team": team,
            "Points": stats["points"],
            "GD": stats["gf"] - stats["ga"],
            "GF": stats["gf"]
        })

    table = sorted(
        table,
        key=lambda x: (-x["Points"], -x["GD"], -x["GF"])
    )

    positions = {}
    i = 0

    while i < len(table):
        j = i

        # Find all teams tied on points, goal difference and goals for
        while (
            j + 1 < len(table)
            and table[j + 1]["Points"] == table[i]["Points"]
            and table[j + 1]["GD"] == table[i]["GD"]
            and table[j + 1]["GF"] == table[i]["GF"]
        ):
            j += 1

        # Average position for tied teams
        average_position = ((i + 1) + (j + 1)) / 2

        for k in range(i, j + 1):
            positions[table[k]["Team"]] = average_position

        i = j + 1

    return positions

In [ ]:
feature_rows = []

# Group by DateTime so that simultaneous matches do not leak information into each other
for datetime_value, matches_at_time in df.groupby("DateTime", sort=True):

    # League positions before any matches at this kick-off time are played
    current_positions = get_league_positions(team_stats)

    # First create features for all matches at this kick-off time
    for idx, row in matches_at_time.iterrows():

        home_team = row["HomeTeam"]
        away_team = row["AwayTeam"]

        home_stats = team_stats[home_team]
        away_stats = team_stats[away_team]

        home_position = current_positions[home_team]
        away_position = current_positions[away_team]

        home_goal_diff_before = home_stats["gf"] - home_stats["ga"]
        away_goal_diff_before = away_stats["gf"] - away_stats["ga"]

        home_points_last5 = last_n_total(home_stats["recent_points"], 5)
        away_points_last5 = last_n_total(away_stats["recent_points"], 5)

        home_avg_points_last5 = last_n_average(home_stats["recent_points"], 5)
        away_avg_points_last5 = last_n_average(away_stats["recent_points"], 5)

        home_goals_scored_last5 = last_n_total(home_stats["recent_gf"], 5)
        away_goals_scored_last5 = last_n_total(away_stats["recent_gf"], 5)

        home_goals_conceded_last5 = last_n_total(home_stats["recent_ga"], 5)
        away_goals_conceded_last5 = last_n_total(away_stats["recent_ga"], 5)

        home_goal_diff_last5 = last_n_total(home_stats["recent_gd"], 5)
        away_goal_diff_last5 = last_n_total(away_stats["recent_gd"], 5)

        home_home_points_last5 = last_n_total(home_stats["recent_home_points"], 5)
        away_away_points_last5 = last_n_total(away_stats["recent_away_points"], 5)

        feature_rows.append({
            "row_index": idx,

            # Matches played before this fixture
            "HomeMatchesPlayedBefore": home_stats["played"],
            "AwayMatchesPlayedBefore": away_stats["played"],

            # League position before the match
            "HomeLeaguePosition": home_position,
            "AwayLeaguePosition": away_position,

            # Positive PositionDiff means the home team is higher in the table
            "PositionDiff": away_position - home_position,

            # Season performance before the match
            "HomeSeasonPointsBefore": home_stats["points"],
            "AwaySeasonPointsBefore": away_stats["points"],
            "SeasonPointsDiff": home_stats["points"] - away_stats["points"],

            "HomeSeasonGoalDiffBefore": home_goal_diff_before,
            "AwaySeasonGoalDiffBefore": away_goal_diff_before,
            "SeasonGoalDiffDiff": home_goal_diff_before - away_goal_diff_before,

            # Recent form: points in last 5 matches
            "HomePointsLast5": home_points_last5,
            "AwayPointsLast5": away_points_last5,
            "PointsLast5Diff": home_points_last5 - away_points_last5,

            # Average recent form
            "HomeAvgPointsLast5": home_avg_points_last5,
            "AwayAvgPointsLast5": away_avg_points_last5,
            "AvgPointsLast5Diff": home_avg_points_last5 - away_avg_points_last5,

            # Recent attacking form
            "HomeGoalsScoredLast5": home_goals_scored_last5,
            "AwayGoalsScoredLast5": away_goals_scored_last5,
            "GoalsScoredLast5Diff": home_goals_scored_last5 - away_goals_scored_last5,

            # Recent defensive form
            "HomeGoalsConcededLast5": home_goals_conceded_last5,
            "AwayGoalsConcededLast5": away_goals_conceded_last5,
            "GoalsConcededLast5Diff": home_goals_conceded_last5 - away_goals_conceded_last5,

            # Recent goal difference form
            "HomeGoalDiffLast5": home_goal_diff_last5,
            "AwayGoalDiffLast5": away_goal_diff_last5,
            "GoalDiffLast5Diff": home_goal_diff_last5 - away_goal_diff_last5,

            # Home/away-specific form
            "HomeHomePointsLast5": home_home_points_last5,
            "AwayAwayPointsLast5": away_away_points_last5,
            "HomeAwaySpecificFormDiff": home_home_points_last5 - away_away_points_last5,

            # Win-rate features before the match
            "HomeWinRateBefore": win_rate(home_stats),
            "AwayWinRateBefore": win_rate(away_stats),
            "WinRateDiff": win_rate(home_stats) - win_rate(away_stats)
        })

    # Then update team statistics after these matches have been played
    for idx, row in matches_at_time.iterrows():

        home_team = row["HomeTeam"]
        away_team = row["AwayTeam"]

        home_goals = row["FTHG"]
        away_goals = row["FTAG"]

        if home_goals > away_goals:
            home_points = 3
            away_points = 0
            home_result = "W"
            away_result = "L"
        elif home_goals < away_goals:
            home_points = 0
            away_points = 3
            home_result = "L"
            away_result = "W"
        else:
            home_points = 1
            away_points = 1
            home_result = "D"
            away_result = "D"

        # Update home team
        team_stats[home_team]["played"] += 1
        team_stats[home_team]["points"] += home_points
        team_stats[home_team]["gf"] += home_goals
        team_stats[home_team]["ga"] += away_goals

        team_stats[home_team]["recent_points"].append(home_points)
        team_stats[home_team]["recent_gf"].append(home_goals)
        team_stats[home_team]["recent_ga"].append(away_goals)
        team_stats[home_team]["recent_gd"].append(home_goals - away_goals)
        team_stats[home_team]["recent_home_points"].append(home_points)

        if home_result == "W":
            team_stats[home_team]["wins"] += 1
        elif home_result == "D":
            team_stats[home_team]["draws"] += 1
        else:
            team_stats[home_team]["losses"] += 1

        # Update away team
        team_stats[away_team]["played"] += 1
        team_stats[away_team]["points"] += away_points
        team_stats[away_team]["gf"] += away_goals
        team_stats[away_team]["ga"] += home_goals

        team_stats[away_team]["recent_points"].append(away_points)
        team_stats[away_team]["recent_gf"].append(away_goals)
        team_stats[away_team]["recent_ga"].append(home_goals)
        team_stats[away_team]["recent_gd"].append(away_goals - home_goals)
        team_stats[away_team]["recent_away_points"].append(away_points)

        if away_result == "W":
            team_stats[away_team]["wins"] += 1
        elif away_result == "D":
            team_stats[away_team]["draws"] += 1
        else:
            team_stats[away_team]["losses"] += 1

In [ ]:
football_features_df = (
    pd.DataFrame(feature_rows)
    .set_index("row_index")
    .sort_index()
)

df = pd.concat([df, football_features_df], axis=1)

df.head()

In [ ]:
football_features = [
    "HomeMatchesPlayedBefore",
    "AwayMatchesPlayedBefore",

    "HomeLeaguePosition",
    "AwayLeaguePosition",
    "PositionDiff",

    "HomeSeasonPointsBefore",
    "AwaySeasonPointsBefore",
    "SeasonPointsDiff",

    "HomeSeasonGoalDiffBefore",
    "AwaySeasonGoalDiffBefore",
    "SeasonGoalDiffDiff",

    "HomePointsLast5",
    "AwayPointsLast5",
    "PointsLast5Diff",

    "HomeAvgPointsLast5",
    "AwayAvgPointsLast5",
    "AvgPointsLast5Diff",

    "HomeGoalsScoredLast5",
    "AwayGoalsScoredLast5",
    "GoalsScoredLast5Diff",

    "HomeGoalsConcededLast5",
    "AwayGoalsConcededLast5",
    "GoalsConcededLast5Diff",

    "HomeGoalDiffLast5",
    "AwayGoalDiffLast5",
    "GoalDiffLast5Diff",

    "HomeHomePointsLast5",
    "AwayAwayPointsLast5",
    "HomeAwaySpecificFormDiff",

    "HomeWinRateBefore",
    "AwayWinRateBefore",
    "WinRateDiff"
]

In [ ]:
df[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "HomeLeaguePosition",
        "AwayLeaguePosition",
        "PositionDiff",
        "HomePointsLast5",
        "AwayPointsLast5",
        "PointsLast5Diff",
        "HomeGoalsScoredLast5",
        "AwayGoalsScoredLast5",
        "HomeGoalsConcededLast5",
        "AwayGoalsConcededLast5",
        "HomeGoalDiffLast5",
        "AwayGoalDiffLast5"
    ]
].head(15)

The football-specific features were created chronologically and include:

- league position before the match,
- season points before the match,
- season goal difference before the match,
- points from the last five matches,
- goals scored and conceded in the last five matches,
- recent goal difference,
- home/away-specific recent form,
- win rate before the match.

This creates a realistic pre-match modelling setup because the features are based only on information that would have been known before kick-off.

## Football-Only Prediction Models

The previous models used Bet365-derived features as predictors. I now train models using only football-specific features.

These features include league position, recent form, season points, goal difference, recent goals scored and conceded, and win rate. This allows us to test whether football performance variables alone can predict match outcomes.

The same modelling approach is used as before: Logistic Regression and Random Forest classification. This keeps the comparison with the Bet365-only models fair.

In [ ]:
# Use only football-specific features
X_football = df[football_features]
y_football_labels = df["FTR"]

# Remove rows with missing football features
football_model_df = pd.concat([X_football, y_football_labels], axis=1).dropna()

X_football = football_model_df[football_features]
y_football_labels = football_model_df["FTR"]

# Encode target variable: A, D, H -> numeric labels
football_encoder = LabelEncoder()
y_football_encoded = football_encoder.fit_transform(y_football_labels)

X_train_football, X_test_football, y_train_football, y_test_football = train_test_split(
    X_football,
    y_football_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_football_encoded
)

print("Training matches:", X_train_football.shape[0])
print("Testing matches:", X_test_football.shape[0])
print("Target classes:", list(football_encoder.classes_))

In [ ]:
# Logistic Regression model

football_log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

football_log_model.fit(X_train_football, y_train_football)

football_log_preds = football_log_model.predict(X_test_football)

football_log_accuracy = accuracy_score(
    y_test_football,
    football_log_preds
)

football_log_f1 = f1_score(
    y_test_football,
    football_log_preds,
    average="macro"
)

print("Football-Only Logistic Regression Accuracy:")
print(round(football_log_accuracy, 4))

print("Football-Only Logistic Regression Macro F1:")
print(round(football_log_f1, 4))

In [ ]:
football_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

football_rf_model.fit(X_train_football, y_train_football)

football_rf_preds = football_rf_model.predict(X_test_football)

football_rf_accuracy = accuracy_score(
    y_test_football,
    football_rf_preds
)

football_rf_f1 = f1_score(
    y_test_football,
    football_rf_preds,
    average="macro"
)

print("Football-Only Random Forest Accuracy:")
print(round(football_rf_accuracy, 4))

print("Football-Only Random Forest Macro F1:")
print(round(football_rf_f1, 4))

In [ ]:
football_only_results = pd.DataFrame({
    "Feature Set": [
        "Football Features Only",
        "Football Features Only"
    ],
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        football_log_accuracy,
        football_rf_accuracy
    ],
    "Macro F1": [
        football_log_f1,
        football_rf_f1
    ]
})

football_only_results

## Football-Only Model Comparison

The football-only models test whether engineered football performance variables can predict outcomes without using bookmaker information.

These features do contain some useful signal: league position, form, points, goals and win rate are all meaningful indicators of team strength. However, football outcomes are noisy, and draws are particularly difficult to predict.

The key comparison is whether these football-only models can match or beat the Bet365-based benchmarks. In this project, the football-only models do not clearly outperform the bookmaker favourite rule, suggesting that the betting market already captures a large amount of the available predictive information.

The next step is to combine Bet365 features with football-specific features to test whether the engineered football variables add incremental value beyond the market odds.

## 9. Combined Bet365 + Football Models

The final modelling stage combines both sources of information:

1. Bet365-derived market features.
2. Chronologically engineered football performance features.

This tests whether football-specific features add predictive value beyond the betting market.

In [ ]:
combined_features = bet365_features + football_features

missing_features = [col for col in combined_features if col not in df.columns]

if len(missing_features) > 0:
    raise ValueError(f"Missing combined features: {missing_features}")

X_combined = df[combined_features]
y_combined_labels = df["FTR"]

combined_model_df = pd.concat([X_combined, y_combined_labels], axis=1).dropna()

X_combined = combined_model_df[combined_features]
y_combined_labels = combined_model_df["FTR"]

combined_encoder = LabelEncoder()
y_combined_encoded = combined_encoder.fit_transform(y_combined_labels)

X_train_combined, X_test_combined, y_train_combined, y_test_combined = train_test_split(
    X_combined,
    y_combined_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_combined_encoded
)

print("Training matches:", X_train_combined.shape[0])
print("Testing matches:", X_test_combined.shape[0])
print("Target classes:", list(combined_encoder.classes_))

In [ ]:
# Combined Logistic Regression model

combined_log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

combined_log_model.fit(X_train_combined, y_train_combined)

combined_log_preds = combined_log_model.predict(X_test_combined)

combined_log_accuracy = accuracy_score(
    y_test_combined,
    combined_log_preds
)

combined_log_f1 = f1_score(
    y_test_combined,
    combined_log_preds,
    average="macro"
)

print("Combined Logistic Regression Accuracy:")
print(round(combined_log_accuracy, 4))

print("Combined Logistic Regression Macro F1:")
print(round(combined_log_f1, 4))

In [ ]:
# Combined Random Forest model

combined_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

combined_rf_model.fit(X_train_combined, y_train_combined)

combined_rf_preds = combined_rf_model.predict(X_test_combined)

combined_rf_accuracy = accuracy_score(
    y_test_combined,
    combined_rf_preds
)

combined_rf_f1 = f1_score(
    y_test_combined,
    combined_rf_preds,
    average="macro"
)

print("Combined Random Forest Accuracy:")
print(round(combined_rf_accuracy, 4))

print("Combined Random Forest Macro F1:")
print(round(combined_rf_f1, 4))

In [ ]:
combined_results = pd.DataFrame({
    "Feature Set": [
        "Bet365 + Football Features",
        "Bet365 + Football Features"
    ],
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        combined_log_accuracy,
        combined_rf_accuracy
    ],
    "Macro F1": [
        combined_log_f1,
        combined_rf_f1
    ]
})

combined_results

## 10. Final Model Comparison

The final comparison brings together all models into one table. Accuracy measures the proportion of correct predictions, while Macro F1 gives a more balanced view across home wins, draws and away wins.

Macro F1 is important here because football match outcomes are not perfectly balanced, and draws are typically harder to predict than home or away wins.

In [ ]:
all_results = pd.concat(
    [
        bet365_results,
        football_only_results,
        combined_results
    ],
    ignore_index=True
)

all_results = all_results.sort_values("Accuracy", ascending=False).reset_index(drop=True)

all_results

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(
    all_results["Feature Set"] + "\n" + all_results["Model"],
    all_results["Accuracy"]
)
plt.title("Final Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
best_model_row = all_results.iloc[0]
print("Best model by accuracy:")
print(best_model_row)

## 11. Confusion Matrix for the Best Machine Learning Model

The simple favourite rule is a strong benchmark, but it is not a trained machine learning model. To inspect the behaviour of a trained classifier, the confusion matrix below uses the best-performing machine learning model from the final modelling stage.

This helps show which outcomes are easier or harder for the model to classify.

In [ ]:
# Select the best trained ML model available in the notebook.
# This avoids using the rule-based favourite benchmark for the confusion matrix.
trained_ml_models = {
    "Bet365 Logistic Regression": (log_accuracy, y_test, log_preds, encoder),
    "Bet365 Random Forest": (rf_accuracy, y_test, rf_preds, encoder),
    "Football Logistic Regression": (football_log_accuracy, y_test_football, football_log_preds, football_encoder),
    "Football Random Forest": (football_rf_accuracy, y_test_football, football_rf_preds, football_encoder),
    "Combined Logistic Regression": (combined_log_accuracy, y_test_combined, combined_log_preds, combined_encoder),
    "Combined Random Forest": (combined_rf_accuracy, y_test_combined, combined_rf_preds, combined_encoder),
}

best_ml_name, (best_ml_accuracy, best_y_true, best_y_pred, best_encoder) = max(
    trained_ml_models.items(),
    key=lambda item: item[1][0]
)

print("Best trained ML model:", best_ml_name)
print("Accuracy:", round(best_ml_accuracy, 4))
print()
print(classification_report(
    best_y_true,
    best_y_pred,
    target_names=best_encoder.classes_
))

ConfusionMatrixDisplay.from_predictions(
    best_y_true,
    best_y_pred,
    display_labels=best_encoder.classes_
)
plt.title(f"Confusion Matrix: {best_ml_name}")
plt.show()

NameError: name 'log_accuracy' is not defined

## 12. Interpretation

The results suggest that the betting market is difficult to beat using the engineered features in this project. The Bet365 favourite rule is a particularly strong benchmark because it directly reflects the market's preferred outcome for each fixture.

The football-only models perform below the bookmaker benchmark, but they still demonstrate that football performance data contains predictive signal. Recent points, league position, goal difference and win rate are all sensible indicators of team strength. However, the signal is noisy, especially because individual football matches are low-scoring and often decided by small margins.

The combined models do not clearly improve performance beyond the Bet365-based benchmarks. This suggests that the bookmaker odds already incorporate much of the information captured by the engineered football features.

## 13. Limitations

This project has several important limitations:

1. **Single-season sample size**: The dataset contains one Premier League season, giving only 380 matches. This is small for a three-class machine learning problem.
2. **Football outcomes are noisy**: Individual matches are affected by red cards, injuries, finishing variance, tactical changes and random events.
3. **Draws are difficult to predict**: Draws are less frequent and often harder to distinguish from narrow home or away wins.
4. **Limited pre-match features**: The engineered football features use past results, goals and form, but do not include richer data such as expected goals, injuries, lineups, rest days, travel distance or player-level information.
5. **Bookmaker odds are already information-rich**: Odds are market prices that aggregate expert knowledge, public information and betting activity. This makes them a very difficult benchmark to outperform.

## 14. Future Improvements

The project could be improved by:

- using multiple Premier League seasons rather than one season,
- adding expected goals data,
- creating Elo-style team ratings,
- including rest days and fixture congestion,
- including injury and lineup information,
- tuning model hyperparameters more systematically,
- testing probability calibration rather than only class prediction,
- evaluating betting profitability using implied probabilities and model probabilities.

A natural next extension would be to move from predicting only the most likely result to estimating the probability of each result and comparing those probabilities against bookmaker implied probabilities. That would allow the project to test betting value more directly.

## 15. Final Conclusion

This project investigated whether Premier League match outcomes could be predicted using bookmaker odds, engineered football performance features, and a combination of both.

The strongest benchmark was the simple Bet365 favourite rule. The football-only models showed that recent form and team-strength features contain useful predictive signal, but they did not clearly outperform the bookmaker-based benchmark. The combined models also did not provide clear evidence that football-specific features added extra predictive value beyond Bet365 odds.

The main conclusion is therefore:

> **For this dataset, the betting market remained more predictive than the engineered football features, and the combined model did not show evidence of beating the information contained in Bet365 odds.**

This supports the idea that bookmaker odds are highly information-rich and that beating the market requires either richer data, more sophisticated modelling, or a focus on probability calibration and value detection rather than simple outcome classification.